# Video Transcription with Speaker Diarization

**Goal:** Transcribe video files with speaker identification using WhisperX

**Steps:**
1. Extract audio from video
2. Run OpenAI whisper for transcription + diarization
3. Output timestamped transcript with speaker labels

In [2]:
# Import packages

import os, torch
from pathlib import Path
from dotenv import load_dotenv

# Load evnironment variables from .env file in project root
load_dotenv()  
HF_TOKEN = os.getenv("HF_TOKEN")

# Device setup
import torch

def get_device(force_cpu=False):
    if force_cpu:
        return "cpu"
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = get_device(force_cpu=True)
DEVICE_PYANNOTE = "mps" if DEVICE == "mps" else DEVICE  # Change to "cpu" if issues

print(f"Whisper device: {DEVICE}")
print(f"Pyannote device: {DEVICE_PYANNOTE}")

Whisper device: cpu
Pyannote device: cpu


In [3]:
# Download video

import subprocess

def download_video(url: str, output_path: str | Path, force: bool = False) -> Path:
    """Download video using wget or curl."""
    output_path = Path(output_path)  # Convert string to Path
    
    if output_path.exists() and not force:
        print(f"Already exists: {output_path}")
        return output_path
    
    output_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["curl", "-L", "-o", str(output_path), url], check=True)
    
    print(f"✅ Downloaded: {output_path} ({output_path.stat().st_size / 1e6:.1f} MB)")
    return output_path
VIDEO_FILE_PATH = "videos/hacks-for-working-with-spreadsheets.mp4"
download_video("https://vod.api.video/vod/vi1E0rcU1EZSJ29BMl8XZdtP/mp4/source.mp4", VIDEO_FILE_PATH)

Already exists: videos/hacks-for-working-with-spreadsheets.mp4


PosixPath('videos/hacks-for-working-with-spreadsheets.mp4')

In [4]:
# Extract audio → 16kHz mono WAV

import subprocess, os
from pathlib import Path

AUDIO_FILE = f"{Path(VIDEO_FILE_PATH).stem}_audio.wav"

subprocess.run([
    "ffmpeg", "-i", VIDEO_FILE_PATH,
    "-vn",                   # no video
    "-acodec", "pcm_s16le",  # 16-bit PCM
    "-ar", "16000",          # 16kHz
    "-ac", "1",              # mono
    "-y", AUDIO_FILE,
], capture_output=True, check=True)

size_mb = os.path.getsize(AUDIO_FILE) / 1e6

probe = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration",
     "-of", "default=noprint_wrappers=1:nokey=1", AUDIO_FILE],
    capture_output=True, text=True,
)
DURATION = float(probe.stdout.strip())

print(f"✓ {AUDIO_FILE} ({size_mb:.1f} MB, {int(DURATION//60)}m {int(DURATION%60)}s)")

✓ hacks-for-working-with-spreadsheets_audio.wav (58.5 MB, 30m 27s)


In [5]:
# Transcribe with Whisper

import whisper, time

WHISPER_MODEL = "medium"  # tiny | base | small | medium | large
LANGUAGE = None       # "en", "fr", etc. or None for auto-detect
NUM_SPEAKERS = 2   # exact count if known, else None
MIN_SPEAKERS = None   # optional lower bound
MAX_SPEAKERS = None   # optional upper bound

print(f"Loading Whisper '{WHISPER_MODEL}' on {DEVICE}...")
model = whisper.load_model(WHISPER_MODEL, device=DEVICE)

print("Transcribing...")
t0 = time.time()

whisper_result = model.transcribe(
    AUDIO_FILE,
    language=LANGUAGE,
    word_timestamps=True,  # per-word timing for alignment
    verbose=False,
)

elapsed = time.time() - t0
seg_count = len(whisper_result["segments"])
word_count = sum(len(s.get("words", [])) for s in whisper_result["segments"])

print(f"✓ Done in {elapsed:.1f}s — {seg_count} segments, {word_count} words, "
      f"lang={whisper_result['language']}, {DURATION/elapsed:.1f}x realtime")

Loading Whisper 'medium' on cpu...
Transcribing...


/Users/rihards/Workspace/fac/prototypes/vectorise-video-transcript/.venv/lib/python3.14/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Detected language: English


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 182745/182745 [03:59<00:00, 761.48frames/s]

✓ Done in 241.0s — 278 segments, 4737 words, lang=en, 7.6x realtime


In [6]:
# Preview whisper output

for seg in whisper_result["segments"][:5]:
    m, s = int(seg["start"] // 60), seg["start"] % 60
    print(f"[{m}:{s:05.2f}] {seg['text'].strip()}")

[0:11.24] Hello, hello and welcome.
[0:14.62] I will give it a couple of seconds for everybody to join.
[0:19.62] Well, hello and welcome to this quickfire masterclass on hacks for working with spreadsheets.
[0:25.62] My name is Victoria and I'm an event producer here at apolitical.
[0:30.66] It would be great to hear from you all so please do post in the chat where you are joining us from


In [8]:
# Diarization

from pyannote.audio import Pipeline as PyannotePipeline


# Load model (downloads on first run)
print("Loading Pyannote...")
diarization_pipeline = PyannotePipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=HF_TOKEN,
)

diarization_pipeline.to(torch.device(DEVICE_PYANNOTE))

# Build kwargs
kwargs = {
    k: v for k, v in {
        "num_speakers": NUM_SPEAKERS,
        "min_speakers": MIN_SPEAKERS,
        "max_speakers": MAX_SPEAKERS,
    }.items() if v is not None
}

print(f"Running diarization...{f' (expecting {kwargs})' if kwargs else ''}")
t0 = time.time()

diarization_result = diarization_pipeline(AUDIO_FILE, **kwargs)

elapsed = time.time() - t0

diarization = diarization_result.speaker_diarization

diarization_turns = [
    {"start": turn.start, "end": turn.end, "speaker": speaker}
    for turn, _, speaker in diarization.itertracks(yield_label=True)
]

speakers = sorted(set(t["speaker"] for t in diarization_turns))
print(f"✓ Done in {elapsed:.1f}s — {len(speakers)} speakers, {len(diarization_turns)} turns")


Loading Pyannote...
Running diarization... (expecting {'num_speakers': 2})


/Users/rihards/Workspace/fac/prototypes/vectorise-video-transcript/.venv/lib/python3.14/site-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  std = sequences.std(dim=-1, correction=1)


✓ Done in 998.1s — 2 speakers, 376 turns


In [43]:
# Preview diarization data

for t in diarization_turns[:10]:
    dur = t["end"] - t["start"]
    m, s = int(t["start"] // 60), t["start"] % 60
    print(f"[{m}:{s:05.2f}] {t['speaker']}  ({dur:.1f}s)")

[0:11.61] SPEAKER_00  (2.3s)
[0:14.71] SPEAKER_00  (4.0s)
[0:19.52] SPEAKER_00  (6.1s)
[0:25.83] SPEAKER_00  (4.4s)
[0:30.61] SPEAKER_00  (7.6s)
[0:38.51] SPEAKER_00  (6.6s)
[0:45.39] SPEAKER_00  (8.2s)
[0:54.55] SPEAKER_00  (9.9s)
[1:05.25] SPEAKER_00  (10.7s)
[1:16.36] SPEAKER_00  (1.7s)


In [44]:
from collections import Counter

aligned_words = []

for segment in whisper_result["segments"]:
    words = segment.get("words", [])
    if not words:
        words = [{"word": segment["text"].strip(),
                  "start": segment["start"], "end": segment["end"]}]

    for w in words:
        w_start = w.get("start", 0)
        w_end = w.get("end", w_start + 0.1)
        w_text = w.get("word", "").strip()
        if not w_text:
            continue

        best_speaker = "UNKNOWN"
        best_overlap = 0.0
        for turn in diarization_turns:
            overlap = max(0.0, min(w_end, turn["end"]) - max(w_start, turn["start"]))
            if overlap > best_overlap:
                best_overlap = overlap
                best_speaker = turn["speaker"]

        aligned_words.append({
            "text": w_text, "start": w_start,
            "end": w_end, "speaker": best_speaker,
        })

print(f"✓ {len(aligned_words)} words assigned")
for spk, count in Counter(w["speaker"] for w in aligned_words).most_common():
    print(f"  {spk}: {count} words")

✓ 4737 words assigned
  SPEAKER_01: 3401 words
  SPEAKER_00: 1290 words
  UNKNOWN: 46 words


In [50]:
# Re-extract ALL words from Whisper result
words = []
for segment in whisper_result["segments"]:
    for w in segment.get("words", []):
        words.append({
            "word": w["word"],
            "start": w["start"],
            "end": w["end"],
            "probability": w.get("probability", 0.0),
        })

print(f"Total words: {len(words)}")
print(f"Time range: {words[0]['start']:.1f}s → {words[-1]['end']:.1f}s")

Total words: 4737
Time range: 11.2s → 1825.9s


In [51]:
# Align words to speakers with tolerance + neighbour fallback

TOLERANCE = 0.5  # seconds

# Pass 1: Assign speakers with tolerance buffer
for word in words:
    best_speaker = "UNKNOWN"
    best_overlap = 0.0
    for turn in diarization_turns:
        overlap = max(0,
            min(word["end"] + TOLERANCE, turn["end"] + TOLERANCE) -
            max(word["start"] - TOLERANCE, turn["start"] - TOLERANCE)
        )
        if overlap > best_overlap:
            best_overlap = overlap
            best_speaker = turn["speaker"]
    word["speaker"] = best_speaker

unknown_after_pass1 = sum(1 for w in words if w["speaker"] == "UNKNOWN")

# Pass 2: Inherit from nearest neighbour
for i, word in enumerate(words):
    if word["speaker"] == "UNKNOWN":
        if i > 0 and words[i - 1]["speaker"] != "UNKNOWN":
            word["speaker"] = words[i - 1]["speaker"]
        elif i < len(words) - 1 and words[i + 1]["speaker"] != "UNKNOWN":
            word["speaker"] = words[i + 1]["speaker"]

unknown_after_pass2 = sum(1 for w in words if w["speaker"] == "UNKNOWN")

print(f"UNKNOWN words: {unknown_after_pass1} (after tolerance) → {unknown_after_pass2} (after neighbour)")
print(f"\nSpeaker breakdown:")
from collections import Counter
counts = Counter(w["speaker"] for w in words)
for speaker, count in sorted(counts.items()):
    print(f"  {speaker}: {count} words")

UNKNOWN words: 0 (after tolerance) → 0 (after neighbour)

Speaker breakdown:
  SPEAKER_00: 1294 words
  SPEAKER_01: 3443 words


In [54]:
segments = []
cur_speaker = words[0]["speaker"]
cur_words = [words[0]]

for word in words[1:]:
    if word["speaker"] == cur_speaker:
        cur_words.append(word)
    else:
        segments.append({
            "start": cur_words[0]["start"],
            "end": cur_words[-1]["end"],
            "speaker": cur_speaker,
            "text": " ".join(w["word"].strip() for w in cur_words),
            "word_count": len(cur_words),
        })
        cur_speaker = word["speaker"]
        cur_words = [word]

segments.append({
    "start": cur_words[0]["start"],
    "end": cur_words[-1]["end"],
    "speaker": cur_speaker,
    "text": " ".join(w["word"].strip() for w in cur_words),
    "word_count": len(cur_words),
})

speakers = sorted(set(s["speaker"] for s in segments))
print(f"✓ {len(segments)} speaker segments, {len(speakers)} speakers")

✓ 17 speaker segments, 2 speakers


In [58]:
def fmt(sec):
    m, s = int(sec // 60), sec % 60
    return f"{m}:{s:05.2f}"

print("=" * 70)
print(f"TRANSCRIPT — {len(speakers)} speakers, {len(segments)} segments")
print("=" * 70 + "\n")

prev = None
for seg in segments:
    if seg["speaker"] != prev:
        if prev: print()
        print(f"── {seg['speaker']} ──")
    print(f"[{fmt(seg['start'])}] {seg['text']}")
    prev = seg["speaker"]

TRANSCRIPT — 2 speakers, 17 segments

── SPEAKER_00 ──
[0:11.24] Hello, hello and welcome. I will give it a couple of seconds for everybody to join. Well, hello and welcome to this quickfire masterclass on hacks for working with spreadsheets. My name is Victoria and I'm an event producer here at apolitical. It would be great to hear from you all so please do post in the chat where you are joining us from and what organization you work for. A technical note that your messages might come through to only our backstage team due to Zoom's default settings so please be sure to select everyone from the blue drop down menu on the zoom chat if you'd like to send a message to all of us here today. Well here today we are looking at how public servants can work more efficiently with spreadsheets. One of the most used but least taught tools in government. We'll explore practical ways to clean your data, fix formulas, structure sheets to save time, reduce errors and take the frustration out of your 

In [57]:
import json

stem = Path(AUDIO_FILE).stem
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

# JSON
output = {
    "source_file": AUDIO_FILE,
    "language": whisper_result["language"],
    "duration_seconds": DURATION,
    "num_speakers": len(speakers),
    "speakers": speakers,
    "segments": [
        {**s, "start_fmt": fmt(s["start"]), "end_fmt": fmt(s["end"])}
        for s in segments
    ],
}
json_path = output_dir / f"{stem}.transcript.json"
with open(json_path, "w") as f:
    json.dump(output, f, indent=2)

# SRT
srt_lines = []
for i, seg in enumerate(segments, 1):
    def srt_ts(sec):
        h, rem = divmod(sec, 3600)
        m, s = divmod(rem, 60)
        ms = int((s - int(s)) * 1000)
        return f"{int(h):02d}:{int(m):02d}:{int(s):02d},{ms:03d}"
    srt_lines += [str(i), f"{srt_ts(seg['start'])} --> {srt_ts(seg['end'])}",
                  f"[{seg['speaker']}] {seg['text']}", ""]
srt_path = output_dir / f"{stem}.srt"
with open(srt_path, "w") as f:
    f.write("\n".join(srt_lines))

# Plain text
txt_path = output_dir / f"{stem}.transcript.txt"
with open(txt_path, "w") as f:
    f.write("\n".join(f"[{fmt(s['start'])}] {s['speaker']}: {s['text']}" for s in segments))

print(f"✓ {json_path}\n✓ {srt_path}\n✓ {txt_path}")

✓ output/hacks-for-working-with-spreadsheets_audio.transcript.json
✓ output/hacks-for-working-with-spreadsheets_audio.srt
✓ output/hacks-for-working-with-spreadsheets_audio.transcript.txt
